# 5-Qubit Star/Hub Chip - Eigenmode Simulation


In [ ]:
!git clone https://github.com/Abdelwhabmohammed/cinqubit.git
%cd cinqubit/src
!pip install qdesignoptimizer
!pip install "quantum-metal[full]"

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Rendering the Design

In [ ]:
import names_star as n
import design_star as d
from qdesignoptimizer.utils.chip_generation import create_chip_base

design, gui = create_chip_base(n.CHIP_NAME, d.chip_type, open_gui=True)
d.render_qiskit_metal_design(design, gui)

## 2. Design Rule Check (DRC)

In [ ]:
import design_rules as dr

violations = dr.run_drc(
    qubit_positions=d.QUBIT_POSITIONS,
    chip_size_x_mm=12.0,
    chip_size_y_mm=12.0,
)
dr.print_drc_report(violations)

## 3. Parameter Sweep - Qubit Pitch Optimization


In [ ]:
import math
import numpy as np
import design_rules as dr

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def generate_star_positions(radius_um):
    """Generate star positions with given radial distance from center."""
    positions = {
        1: {"pos_x": f"{-radius_um/1000}mm", "pos_y": f"{-radius_um/1000}mm", "orientation": "0"},
        2: {"pos_x": f"{radius_um/1000}mm",  "pos_y": f"{-radius_um/1000}mm", "orientation": "180"},
        3: {"pos_x": "0.0mm",                  "pos_y": "0.0mm",                 "orientation": "0"},
        4: {"pos_x": f"{-radius_um/1000}mm", "pos_y": f"{radius_um/1000}mm",  "orientation": "0"},
        5: {"pos_x": f"{radius_um/1000}mm",  "pos_y": f"{radius_um/1000}mm",  "orientation": "180"},
    }
    return positions

radii = np.linspace(1500, 4000, 20)
costs = []
drc_pass = []

for r in radii:
    pos = generate_star_positions(r)
    # Distance from center to each corner qubit = r*sqrt(2)
    coupler_length = r * math.sqrt(2)
    total = 4 * coupler_length
    cost = total + 2.0 * coupler_length
    costs.append(cost)
    
    violations = dr.run_drc(pos, 12.0, 12.0)
    drc_pass.append(len(violations) == 0)

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True,
                                    gridspec_kw={"height_ratios": [3, 1]})
    ax1.plot(radii, costs, 'b-o', linewidth=2, markersize=4)
    ax1.set_ylabel('Cost (um)')
    ax1.set_title('5-Qubit Star: Layout Cost vs. Radial Distance')
    ax1.grid(True, alpha=0.3)
    
    colors = ['green' if p else 'red' for p in drc_pass]
    ax2.bar(radii, [1]*len(radii), width=(radii[1]-radii[0])*0.8, color=colors, alpha=0.7)
    ax2.set_ylabel('DRC')
    ax2.set_xlabel('Radial Distance (um)')
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['FAIL', 'PASS'])
    plt.tight_layout()
    plt.savefig('out/radial_sweep_star.png', dpi=150, bbox_inches='tight')
    plt.show()

optimal_idx = np.argmin(costs)
print(f"Minimum cost radius: {radii[optimal_idx]:.0f} um (DRC: {'PASS' if drc_pass[optimal_idx] else 'FAIL'})")

## 4. Creating Study and Optimization Targets


In [ ]:
print("Star topology mini study setup would follow the same pattern as linear chain.")
print(f"Coupling edges: {n.COUPLING_EDGES}")
print(f"Qubits: {n.ALL_QUBITS}")
print(f"Couplers: {n.ALL_COUPLERS}")

## 5. Topology Comparison

In [ ]:
import comparison as comp

candidates = [comp.LINEAR_CHAIN, comp.STAR_HUB]
df = comp.build_comparison_table(candidates)
if df is not None:
    display(df)

print()
print(comp.make_recommendation(candidates))

## 6. Close

In [ ]:
from qdesignoptimizer.utils.utils import close_ansys
close_ansys()